# Lección 10 · Varias metas: ponderado, lexicográfico y frontera de Pareto

**Qué vas a aprender**
- **Dominancia** y **frontera de Pareto**: las soluciones donde no se mejora una meta sin empeorar otra.
- **Objetivo ponderado**: los pesos son tasas de cambio escondidas. Y hay puntos de la frontera que **ningún** peso encuentra.
- **Lexicográfico**: optimizar en orden de prioridad, con una tolerancia $\varepsilon$. La trampa: hay que proteger cada meta que importa.
- **ε-constraint**: recorrer la frontera entera, punto por punto.
- **Balance min-max**: qué controla y qué no.
- El modelo crece: tu asignación de cartera (estilo Banco Agrario) con 1.200 clientes, 22 asesores y 3 metas.
- Escribir cada modelo en **notación matemática** y leerla símbolo por símbolo.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno

In [ ]:
%pip install -q pyomo highspy

In [ ]:
import io
import itertools
import logging
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.repn import generate_standard_repn

SOLVER = pyo.SolverFactory("appsi_highs")
logging.getLogger("pyomo.contrib.appsi").setLevel(logging.ERROR)


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def tamano(m):
    # El tamaño del modelo: variables, restricciones y no ceros (coeficientes distintos de 0)
    variables = list(m.component_data_objects(pyo.Var))
    restricciones = list(m.component_data_objects(pyo.Constraint, active=True))
    return {"variables": len(variables),
            "binarias": sum(v.is_binary() for v in variables),
            "restricciones": len(restricciones),
            "no_ceros": sum(sum(c != 0 for c in generate_standard_repn(r.body).linear_coefs) for r in restricciones)}

## 1. Dónde quedamos
En la lección 3 elegiste una cartera de proyectos con **una** meta: el valor presente neto (VPN). En la lección 4 asignaste ingenieros a proyectos, con la estructura de tu modelo de Banco Agrario.
En la vida real casi nunca hay una sola meta. El directorio quiere VPN **y** reducir emisiones. El área comercial quiere valor **y** asesores con carga pareja **y** que el cliente siga con su asesor.

**Analogía.** Comprar el pasaje de tren Kraków–Varsovia. Hay dos metas: precio y duración.
- El Pendolino de 2 h 20 cuesta 169 zł; el TLK de 3 h, 79 zł. Ninguno es mejor en las dos cosas.
- Un tren de 3 h 30 que cuesta 120 zł no tiene sentido: el TLK es más barato **y** más rápido. Ese tren está **dominado**.
- La **frontera de Pareto** es la lista de pasajes no dominados. Tú no eliges por tu esposa cuánto vale una hora en zlotys: le muestras la lista y ella elige.

## 2. Cartera con dos metas: VPN y CO₂
Una empresa de energía tiene 8 proyectos candidatos y **19 M€** de presupuesto. Cada proyecto tiene su VPN y las toneladas de CO₂ que evita por año.

**Las 3 piezas:**
- **Decido:** qué proyectos hago (sí/no).
- **Objetivo:** maximizar el VPN **y** maximizar el CO₂ evitado. Dos metas.
- **Restricciones:** el costo total no pasa del presupuesto.

In [ ]:
PROYECTOS = pd.read_csv(io.StringIO('''proyecto,costo,vpn,co2
eolico_A,7,8.2,176
eolico_marino_B,8,4.2,251
solar_C,3,4.4,102
solar_D,3,5.2,113
baterias_E,7,7.2,71
repotenciacion_F,7,13.5,218
subestacion_G,5,8.5,49
biogas_H,3,2.6,110
'''))
PRESUPUESTO = 19
print("Costo en M€, VPN en M€, CO₂ en kt evitadas por año. Presupuesto:", PRESUPUESTO, "M€")
PROYECTOS.assign(vpn_por_meuro=(PROYECTOS["vpn"] / PROYECTOS["costo"]).round(2), co2_por_meuro=(PROYECTOS["co2"] / PROYECTOS["costo"]).round(1))

**Mira las dos últimas columnas.** La repotenciación (cambiar turbinas viejas por nuevas) es la mejor en VPN por euro. El eólico marino es de los mejores en CO₂ por euro, pero de los peores en VPN.
Las metas **chocan**: no hay una cartera que gane en las dos.

### 2.1 Fuerza bruta: todas las carteras
Hay $2^8 = 256$ carteras. Calculamos el VPN y el CO₂ de las que caben en el presupuesto.

In [ ]:
filas = []
for x in itertools.product([0, 1], repeat=len(PROYECTOS)):
    x = np.array(x)
    if (PROYECTOS["costo"] * x).sum() <= PRESUPUESTO:
        filas.append({"cartera": "+".join(PROYECTOS["proyecto"][x == 1].str.split("_").str[-1]),
                      "costo": int((PROYECTOS["costo"] * x).sum()), "vpn": round((PROYECTOS["vpn"] * x).sum(), 1),
                      "co2": int((PROYECTOS["co2"] * x).sum())})
carteras = pd.DataFrame(filas)
print("Carteras que caben:", len(carteras), "de", 2 ** 8)
carteras.nlargest(3, "vpn")

### En notación: dominancia y frontera
Con dos metas que se maximizan, $f_1$ (VPN) y $f_2$ (CO₂), la cartera $x$ **domina** a $x'$ si es al menos igual en las dos y mejor en una:

$$
x \succ x' \iff f_1(x) \ge f_1(x'),\ \ f_2(x) \ge f_2(x') \ \text{ y } \ \big( f_1(x) > f_1(x') \ \lor\ f_2(x) > f_2(x') \big)
$$

La **frontera de Pareto** es el conjunto de carteras factibles que nadie domina:

$$
\mathcal{P} = \{\, x \in X : \nexists\, x' \in X \text{ con } x' \succ x \,\}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $f_1(x)$, $f_2(x)$ | "efe uno de x" | el valor de cada meta para la solución $x$ | `vpn`, `co2` |
| $x \succ x'$ | "x domina a x prima" | igual o mejor en todo, mejor en algo | `(a >= b).all() and (a > b).any()` |
| $\iff$ | "si y solo si" | las dos frases dicen lo mismo | `==` entre condiciones |
| $\nexists$ | "no existe" | ningún elemento cumple | `not any(...)` |
| $X$ | "equis mayúscula" | el conjunto de soluciones factibles | `carteras` |
| $\mathcal{P}$ | "pe caligráfica" | la frontera de Pareto | `frontera` |

In [ ]:
def no_dominadas(df, metas=("vpn", "co2")):
    valores = df[list(metas)].to_numpy()
    domina = lambda a, b: (a >= b).all() and (a > b).any()
    return df[[not any(domina(otro, este) for otro in valores) for este in valores]]


frontera = no_dominadas(carteras).sort_values("vpn")
frontera

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(carteras["vpn"], carteras["co2"], color="#cbd5e1", s=18, label="cartera factible")
ax.plot(frontera["vpn"], frontera["co2"], "o-", color="#2563eb", label="frontera de Pareto")
for r in frontera.itertuples():
    ax.annotate(r.cartera, (r.vpn, r.co2), textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.set(xlabel="VPN (M€)", ylabel="CO₂ evitado (kt/año)", title="103 carteras que caben en 19 M€")
ax.legend()
plt.tight_layout()
plt.show()

**Lectura:**
- De 103 carteras factibles, solo **5** están en la frontera. Las otras 98 están dominadas: hay otra más rentable **y** más verde.
- Un extremo: **22,9 M€ y 582 kt** (máximo CO₂). El otro: **31,6 M€ y 482 kt** (máximo VPN).
- Entre los dos hay 3 carteras intermedias. Cada una es una **respuesta distinta** a "¿cuánto VPN cedo por tonelada de CO₂?".
- La frontera no es "la solución": es el **menú** para el directorio.

### 2.2 El modelo en Pyomo
Un solo modelo sirve para todo lo que viene: le pasamos qué meta maximizar y, si queremos, un **piso** para cada meta.

$$
\begin{aligned}
\max \quad & f_k(x) && k \in \{1, 2\} \\
\text{s.a.} \quad & \sum_{i \in I} c_i\, x_i \le B && \text{(presupuesto)} \\
& f_1(x) = \sum_{i \in I} v_i\, x_i \ge \ell_1, \quad f_2(x) = \sum_{i \in I} e_i\, x_i \ge \ell_2 && \text{(pisos, opcionales)} \\
& x_i \in \{0, 1\} && \forall\, i \in I
\end{aligned}
$$

In [ ]:
def cartera(meta, pisos=None, pesos=None):
    # meta: "vpn" o "co2". pisos: {meta: mínimo}. pesos: {meta: peso} para el objetivo ponderado (ignora meta).
    I = list(PROYECTOS["proyecto"])
    dato = PROYECTOS.set_index("proyecto")
    m = pyo.ConcreteModel()
    m.x = pyo.Var(I, within=pyo.Binary)
    m.f = {k: sum(dato.loc[i, k] * m.x[i] for i in I) for k in ["vpn", "co2"]}
    expr = sum(pesos[k] * m.f[k] for k in pesos) if pesos else m.f[meta]
    m.obj = pyo.Objective(expr=expr, sense=pyo.maximize)
    m.presupuesto = pyo.Constraint(expr=sum(dato.loc[i, "costo"] * m.x[i] for i in I) <= PRESUPUESTO)
    m.pisos = pyo.ConstraintList()
    for k, minimo in (pisos or {}).items():
        m.pisos.add(m.f[k] >= minimo - 1e-6)
    try:
        res = SOLVER.solve(m)
    except RuntimeError:  # HiGHS no encontró ninguna solución que cargar
        return None
    if res.solver.termination_condition != pyo.TerminationCondition.optimal:
        return None  # infactible: ninguna cartera cumple los pisos
    elegidos = [i for i in I if pyo.value(m.x[i]) > 0.5]
    return {"cartera": "+".join(i.split("_")[-1] for i in elegidos),
            "vpn": round(pyo.value(m.f["vpn"]), 1), "co2": round(pyo.value(m.f["co2"])), "modelo": m}


solo_vpn, solo_co2 = cartera("vpn"), cartera("co2")
print("Solo VPN:", {k: v for k, v in solo_vpn.items() if k != "modelo"})
print("Solo CO₂:", {k: v for k, v in solo_co2.items() if k != "modelo"})
print("Tamaño:", tamano(solo_vpn["modelo"]))

**Lectura:**
- Con una sola meta, el modelo da uno de los **extremos** de la frontera. Cada uno ignora por completo la otra meta.
- Es la mochila de la lección 3: 8 binarias, 1 restricción.

**Los dos tamaños.** Modelo: $\lvert I \rvert$ binarias. Espacio de búsqueda: $2^{\lvert I \rvert}$ carteras. Lo nuevo es un tercer tamaño:
**la frontera**, que puede tener desde 1 punto hasta tantos como carteras. Cada punto cuesta un MILP.

## 3. Objetivo ponderado: los pesos son tasas de cambio
La forma más común: sumar las metas con pesos.

$$
\max \quad \lambda\, \frac{f_1(x)}{f_1^*} + (1 - \lambda)\, \frac{f_2(x)}{f_2^*}, \qquad \lambda \in [0, 1]
$$

Se divide cada meta por su mejor valor ($f_1^* = 31{,}6$ M€, $f_2^* = 582$ kt) para que las dos queden en la misma escala, entre 0 y 1.
Sin eso, la meta con números más grandes manda sola.

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\lambda$ | "lambda" | cuánto pesa la meta 1; la meta 2 pesa $1 - \lambda$ | `lam` |
| $f_k^*$ | "efe estrella" | el mejor valor posible de la meta $k$ sola | `solo_vpn["vpn"]` |
| $f_k(x) / f_k^*$ | "normalizada" | qué fracción del mejor valor logra $x$ | `vpn / vpn_max` |

**La tasa de cambio escondida.** Con $\lambda = 0{,}5$, una tonelada vale lo mismo que $31{,}6 / 582 = 0{,}054$ M€: **54.000 € por kt**. Nadie lo dijo, pero el modelo lo decidió.

Barremos $\lambda$ de 0 a 1 en 101 pasos y miramos qué carteras salen.

In [ ]:
F1, F2 = solo_vpn["vpn"], solo_co2["co2"]
ponderado = []
for lam in np.linspace(0, 1, 101):
    r = cartera(None, pesos={"vpn": lam / F1, "co2": (1 - lam) / F2})
    ponderado.append({"lambda": round(lam, 2), "cartera": r["cartera"], "vpn": r["vpn"], "co2": r["co2"]})
ponderado = pd.DataFrame(ponderado)
ponderado.groupby(["cartera", "vpn", "co2"])["lambda"].agg(["min", "max", "count"]).reset_index().sort_values("vpn")

In [ ]:
encontradas = set(ponderado["cartera"])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(carteras["vpn"], carteras["co2"], color="#cbd5e1", s=18)
ax.plot(frontera["vpn"], frontera["co2"], "o-", color="#2563eb", label="frontera de Pareto")
alcanzadas = frontera[frontera["cartera"].isin(encontradas)]
ax.plot(alcanzadas["vpn"], alcanzadas["co2"], "s--", color="#dc2626", ms=10, mfc="none", label="lo que encuentra el ponderado")
ax.set(xlabel="VPN (M€)", ylabel="CO₂ evitado (kt/año)", title="101 pesos distintos, 2 carteras")
ax.legend()
plt.tight_layout()
plt.show()

**Lectura:**
- Con 101 pesos distintos, el ponderado solo encuentra **2 carteras**: los dos extremos. Salta de una a otra entre $\lambda = 0{,}38$ y $0{,}39$.
- Las 3 carteras del medio **nunca** salen, con ningún peso. Están "hundidas": quedan por debajo de la recta que une los extremos.
  Se llaman puntos **no soportados** (*unsupported*). Con variables enteras pasa seguido.
- Si el directorio quería algo intermedio, por ejemplo **26,9 M€ y 507 kt**, el ponderado no se lo puede dar.

**Tres problemas del ponderado:**
1. Los pesos son tasas de cambio que nadie decidió de frente.
2. Un cambio chico de peso puede saltar de un extremo al otro.
3. Con enteros, no llega a toda la frontera.

## 4. Lexicográfico: en orden de prioridad
En vez de pesos, un **orden**: primero el VPN; después, sin perder (casi) nada de VPN, el máximo CO₂.

$$
\begin{aligned}
\text{Paso 1:} \quad & f_1^* = \max_{x \in X} f_1(x) \\
\text{Paso 2:} \quad & \max_{x \in X} f_2(x) \quad \text{s.a.} \quad f_1(x) \ge (1 - \varepsilon)\, f_1^*
\end{aligned}
$$

$\varepsilon$ es cuánto de la meta 1 aceptas ceder: 0 = nada, 0,05 = 5 %.

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\varepsilon$ | "épsilon" | tolerancia: fracción de la meta 1 que se puede ceder | `eps` |
| $(1 - \varepsilon) f_1^*$ | "uno menos épsilon por efe estrella" | el **piso** de la meta 1 en el paso 2 | `(1 - eps) * F1` |
| $\text{lexmax}$ | "máximo lexicográfico" | maximizar en orden, como un diccionario ordena palabras | — |

**Analogía.** Buscar apartamento en Kraków. Paso 1: el arriendo más barato, 3.000 zł. Paso 2: entre los que cuestan como máximo 2 % más (3.060 zł), el más cerca del *przedszkole* de tu hijo.
Nunca decidiste cuántos zlotys vale un minuto de caminata.

In [ ]:
lex = []
for eps in [0, 0.05, 0.10, 0.20, 0.30]:
    r = cartera("co2", pisos={"vpn": (1 - eps) * F1})
    lex.append({"eps": f"{eps:.0%}", "piso_vpn": round((1 - eps) * F1, 1), "cartera": r["cartera"], "vpn": r["vpn"], "co2": r["co2"]})
pd.DataFrame(lex)

**Lectura:**
- Con $\varepsilon = 0$ sale el extremo de máximo VPN: 31,6 M€ y 482 kt. Con 5 % **no cambia**: el piso queda en 30,0 M€ y la siguiente cartera tiene 29,8.
- Con 10 %: 29,8 M€ y **490 kt**. Con 20 %: 25,7 M€ y 543 kt.
- **Aparecen carteras del medio** que el ponderado nunca encontró. Cada $\varepsilon$ es una pregunta concreta al directorio: "¿cederían 10 % del VPN por 8 kt más?".
- Pero con $\varepsilon$ redondos **se salta** la cartera de 26,9 M€ y 507 kt: solo sale con $\varepsilon$ entre 15 % y 18 %. Elegir $\varepsilon$ a ojo no garantiza ver toda la frontera.

## 5. ε-constraint: la frontera entera
Si el lexicográfico con varios $\varepsilon$ encuentra puntos del medio, se puede **recorrer** la frontera completa:
1. Maximiza el VPN sin piso de CO₂. Guarda el punto.
2. Pon el piso de CO₂ un poco **por encima** del último punto y vuelve a maximizar el VPN.
3. Repite hasta que no haya solución.

$$
\max_{x \in X} f_1(x) \quad \text{s.a.} \quad f_2(x) \ge \varepsilon_k, \qquad \varepsilon_{k+1} = f_2(x_k) + \delta
$$

Con metas enteras (aquí, kt enteras), $\delta = 1$ no se salta ningún punto. Cada punto de la frontera cuesta un MILP.

In [ ]:
puntos, piso_co2 = [], 0
while True:
    r = cartera("vpn", pisos={"co2": piso_co2})
    if r is None:  # ninguna cartera llega a ese CO₂: se acabó la frontera
        break
    # desempate: entre las de este VPN, la de más CO₂ (si no, podría salir un punto dominado)
    r = cartera("co2", pisos={"vpn": r["vpn"], "co2": piso_co2})
    puntos.append({"cartera": r["cartera"], "vpn": r["vpn"], "co2": r["co2"]})
    piso_co2 = r["co2"] + 1
pd.DataFrame(puntos)

**Lectura:**
- El ε-constraint encuentra las **5 carteras** de la frontera, las mismas de la fuerza bruta. Con 5 + 5 MILP (cada punto lleva un desempate).
- **El desempate importa.** Si hay dos carteras con el mismo VPN máximo, sin el segundo paso podría salir la de menos CO₂, que está dominada.
- Con 2 metas, este método es el que se usa para mostrarle la curva a negocio. Con 3 o más, la frontera es una superficie: se fijan pisos para todas menos una.

## 6. El modelo crece: tu cartera de Banco Agrario, con 3 metas
Tu modelo real asignaba clientes de cartera a asesores. Aquí, una versión con datos simulados (semilla fija), con la misma estructura:
- **1.200 clientes** con región, etapa de mora, segmento y saldo (en pesos colombianos).
- **22 asesores**: 6 Senior (2 por región), 6 Gestores y 10 Junior. Cada tipo atiende ciertos clientes y tiene capacidad total y por etapa.
  Algunos pueden estar de vacaciones (capacidad a la mitad) o con licencia (capacidad 0). En esta muestra, un Senior está a media capacidad.
- **3 metas:**
  1. **Valor:** saldo × peso del tipo de asesor (Senior 3, Gestor 2, Junior 1) × bono de 50 % si el cliente sigue con su asesor anterior.
  2. **Balance:** que los Junior queden con carga pareja en los segmentos micro.
  3. **Continuidad:** que el cliente siga con su asesor anterior.

El valor ya incluye la continuidad (el bono). El balance es la meta que choca.

In [ ]:
rng = np.random.default_rng(23)
N_CLIENTES = 1200
REGIONES = ["NORTE", "CENTRO", "SUR"]
ETAPAS = ["Temprana", "Media", "Tardia"]
SEGMENTOS = ["MICRO_A", "MICRO_B", "PYME", "EMPRESA"]
SEGMENTOS_MICRO = ["MICRO_A", "MICRO_B"]
PESO = {"Senior": 3, "Gestor": 2, "Junior": 1}
BONO_CONTINUIDAD = 0.5
CAP_TOTAL = {"Senior": 70, "Gestor": 90, "Junior": 110}
CAP_ETAPA = {"Senior": {"Temprana": 30, "Media": 40, "Tardia": 30},
             "Gestor": {"Media": 60, "Tardia": 50},
             "Junior": {"Temprana": 80, "Media": 60}}

asesores = pd.DataFrame([(f"S{i:02d}", "Senior", REGIONES[i % 3]) for i in range(6)]
                        + [(f"G{i:02d}", "Gestor", None) for i in range(6)]
                        + [(f"J{i:02d}", "Junior", None) for i in range(10)], columns=["asesor", "tipo", "region"])
asesores["factor"] = rng.choice([1.0, 0.5, 0.0], len(asesores), p=[0.8, 0.15, 0.05])  # 1 normal, 0,5 vacaciones, 0 licencia
clientes = pd.DataFrame({"cliente": [f"C{i:04d}" for i in range(N_CLIENTES)],
                         "region": rng.choice(REGIONES, N_CLIENTES),
                         "etapa": rng.choice(ETAPAS, N_CLIENTES, p=[0.45, 0.35, 0.20]),
                         "segmento": rng.choice(SEGMENTOS, N_CLIENTES, p=[0.35, 0.25, 0.20, 0.20]),
                         "saldo": np.round(rng.lognormal(np.log(3e6), 1.0, N_CLIENTES), -3)})
print("Asesores por tipo y disponibilidad:")
asesores.pivot_table(index="tipo", columns="factor", values="asesor", aggfunc="count", fill_value=0)

Los **pares factibles** salen de un *cross join* con filtros, como en SQL (lección 4): Senior solo en su región; Gestor solo en etapas media y tardía; Junior solo micro en etapas temprana y media.
Después, el asesor anterior de cada cliente: 60 % tiene uno factible, 10 % uno que ya no puede atenderlo, 30 % ninguno.

In [ ]:
pares = clientes.merge(asesores, how="cross", suffixes=("_c", "_a"))
senior = (pares["tipo"] == "Senior") & (pares["region_c"] == pares["region_a"])
gestor = (pares["tipo"] == "Gestor") & pares["etapa"].isin(["Media", "Tardia"])
junior = (pares["tipo"] == "Junior") & pares["segmento"].isin(SEGMENTOS_MICRO) & pares["etapa"].isin(["Temprana", "Media"])
pares = pares[senior | gestor | junior].reset_index(drop=True)

factibles = pares.groupby("cliente")["asesor"].apply(list)
anterior = {}
for c in clientes["cliente"]:
    u = rng.random()
    if u < 0.6:
        anterior[c] = str(rng.choice(factibles[c]))
    elif u < 0.7:
        anterior[c] = str(rng.choice(asesores["asesor"]))
clientes["anterior"] = clientes["cliente"].map(anterior)
pares["cont"] = pares["asesor"] == pares["cliente"].map(anterior)
CONT_FACTIBLE = pares.loc[pares["cont"], "cliente"].nunique()
print(f"Pares factibles: {len(pares):,} de {N_CLIENTES * len(asesores):,} | clientes con asesor anterior factible: {CONT_FACTIBLE}".replace(",", "."))
pares.head()

### En notación
Conjuntos: $C$ = clientes, $A$ = asesores, $P \subseteq A \times C$ = pares factibles, $E$ = etapas, $J \subseteq A$ = Junior, $S_\mu$ = segmentos micro.
Datos: $s_c$ = saldo, $\omega_a$ = peso del tipo de asesor, $\kappa_{ac} \in \{0, 1\}$ = 1 si $a$ era el asesor anterior de $c$, $u_a$ y $u_{ae}$ = capacidades (ya con vacaciones).
Variables: $x_{ac} \in \{0, 1\}$, y $z_\sigma \ge 0$ = la carga máxima de un Junior en el segmento $\sigma$.

$$
\begin{aligned}
V(x) &= \sum_{(a, c) \in P} s_c\, \omega_a\, (1 + 0{,}5\, \kappa_{ac})\, x_{ac} && \text{(meta 1: valor)} \\
B(z) &= \sum_{\sigma \in S_\mu} z_\sigma && \text{(meta 2: desbalance, se minimiza)} \\
\text{s.a.} \quad & \sum_{a : (a, c) \in P} x_{ac} \le 1 && \forall\, c \in C \\
& \sum_{c : (a, c) \in P} x_{ac} \le u_a, \qquad \sum_{c \in e : (a, c) \in P} x_{ac} \le u_{ae} && \forall\, a \in A,\ e \in E \\
& \sum_{c \in \sigma : (a, c) \in P} x_{ac} \le z_\sigma && \forall\, a \in J,\ \sigma \in S_\mu \quad \text{(min-max)}
\end{aligned}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\omega_a$ | "omega de a" | peso del tipo: Senior 3, Gestor 2, Junior 1 | `PESO[tipo]` |
| $\kappa_{ac}$ | "kappa" | continuidad: el cliente sigue con su asesor | `pares["cont"]` |
| $z_\sigma \ge$ cargas | "zeta de sigma" | **min-max**: $z$ queda en la carga del Junior más cargado | `m.z[s]` |
| $c \in e$, $c \in \sigma$ | "c en la etapa e" | clientes de esa etapa o segmento | `groupby("etapa")` |

In [ ]:
def capacidad(tipo, factor, etapa=None):
    base = CAP_TOTAL[tipo] if etapa is None else CAP_ETAPA[tipo][etapa]
    return math.ceil(base * factor)


def construir(escala_saldo=1.0):
    # El valor V(x) queda como Expression; el objetivo se agrega afuera según el método.
    P = list(zip(pares["asesor"], pares["cliente"]))
    coef = pares["saldo"] / escala_saldo * pares["tipo"].map(PESO) * (1 + BONO_CONTINUIDAD * pares["cont"])
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=P, dimen=2)
    m.x = pyo.Var(m.P, within=pyo.Binary)
    m.z = pyo.Var(SEGMENTOS_MICRO, within=pyo.NonNegativeReals)
    m.valor = pyo.Expression(expr=pyo.quicksum(k * m.x[p] for p, k in zip(P, coef)))
    m.continuidad = pyo.Expression(expr=pyo.quicksum(m.x[p] for p, cont in zip(P, pares["cont"]) if cont))
    m.unico = pyo.ConstraintList()
    for _, g in pares.groupby("cliente"):
        m.unico.add(pyo.quicksum(m.x[a, c] for a, c in zip(g["asesor"], g["cliente"])) <= 1)
    m.cap = pyo.ConstraintList()
    for (a, t, f), g in pares.groupby(["asesor", "tipo", "factor"]):
        m.cap.add(pyo.quicksum(m.x[a, c] for c in g["cliente"]) <= capacidad(t, f))
        for e, ge in g.groupby("etapa"):
            m.cap.add(pyo.quicksum(m.x[a, c] for c in ge["cliente"]) <= capacidad(t, f, e))
    m.balance = pyo.ConstraintList()
    for (a, s), g in pares[pares["tipo"] == "Junior"].groupby(["asesor", "segmento"]):
        m.balance.add(pyo.quicksum(m.x[a, c] for c in g["cliente"]) <= m.z[s])
    return m


def resolver(m):
    inicio = time.perf_counter()
    SOLVER.solve(m)
    return time.perf_counter() - inicio


def metricas(m, version, segundos):
    x = np.array([pyo.value(m.x[a, c]) > 0.5 for a, c in zip(pares["asesor"], pares["cliente"])])
    asig = pares[x]
    valor = (asig["saldo"] * asig["tipo"].map(PESO) * (1 + BONO_CONTINUIDAD * asig["cont"])).sum()
    jun = asig[asig["tipo"] == "Junior"]
    activos = asesores.loc[(asesores["tipo"] == "Junior") & (asesores["factor"] > 0), "asesor"]
    carga = jun.groupby("asesor").size().reindex(activos, fill_value=0)
    return {"version": version, "segundos": round(segundos, 1), "valor_miles_de_millones": round(valor / 1e9, 3),
            "Σz": int(jun.groupby(["asesor", "segmento"]).size().groupby(level="segmento").max().sum()),
            "carga_junior": f"{carga.min()}-{carga.max()}",
            "continuidad_%": round(100 * asig["cont"].sum() / CONT_FACTIBLE, 1),
            "sin_asignar": N_CLIENTES - asig["cliente"].nunique()}


m_tam = construir()
print("Tamaño:", tamano(m_tam))

**Los dos tamaños.**
- Modelo: una binaria por par factible. El *join* con filtros deja 11.964 de 26.400.
- Espacio de búsqueda: cada cliente elige uno de sus asesores factibles o ninguno: $\prod_{c \in C} (k_c + 1)$, con $k_c$ = asesores factibles de $c$.

In [ ]:
k_c = pares.groupby("cliente").size()
print(f"Asesores factibles por cliente: media {k_c.mean():.1f}, mín {k_c.min()}, máx {k_c.max()}")
print(f"Espacio de búsqueda: 10^{np.log10(k_c + 1).sum():.0f} asignaciones (sin mirar capacidades)")

### 6.1 Ponderado: el balance solo desempata
Tu versión de producción usaba un ponderado: $\max\ V(x) - \lambda \cdot B(z)$, con $\lambda = 1.000$.

In [ ]:
resultados = []
m = construir()
m.obj = pyo.Objective(expr=m.valor - 1000 * pyo.quicksum(m.z.values()), sense=pyo.maximize)
resultados.append(metricas(m, "ponderado λ = 1.000", resolver(m)))
mediana = (pares["saldo"] * pares["tipo"].map(PESO)).median()
print(f"Valor mediano de una asignación: {mediana:,.0f} pesos | λ por unidad de carga: 1.000 | razón {mediana / 1000:,.0f} a 1".replace(",", "."))
pd.DataFrame(resultados)

**Lectura:**
- Una asignación mediana vale unos **4,9 millones** de pesos. Una unidad de carga máxima cuesta 1.000. El balance pesa **5.000 veces menos**: solo desempata.
- La tasa de cambio escondida: el modelo cambia un cliente asignado por 4.900 unidades de balance. Nadie en el banco eligió eso.
- Y si el solver para con un gap de 0,01 %, ese gap (unos 2 millones de pesos) es más grande que todo el término de balance: el balance puede quedar al azar.

### 6.2 Lexicográfico: valor primero, balance después
Saldo en **millones** de pesos para que la fila $V(x) \ge (1 - \varepsilon) V^*$ quede bien escalada (coeficientes de 1 a 100, no de $10^6$ a $10^8$).

In [ ]:
m = construir(escala_saldo=1e6)
m.obj1 = pyo.Objective(expr=m.valor, sense=pyo.maximize)
seg = resolver(m)
V_ESTRELLA = pyo.value(m.valor)
N_ASIGNADOS = sum(pyo.value(v) > 0.5 for v in m.x.values())
CONT_PASO1 = pyo.value(m.continuidad)
resultados.append(metricas(m, "lex paso 1: solo valor", seg))
print(f"V* = {V_ESTRELLA:,.0f} millones | asignados: {N_ASIGNADOS}".replace(",", "."))
pd.DataFrame(resultados)

El paso 2 minimiza el desbalance con el valor protegido por un piso. Y algo más: un **piso de cobertura**, "asigna al menos tantos clientes como el paso 1". Mira por qué en la última fila.

In [ ]:
m.obj1.deactivate()
m.obj2 = pyo.Objective(expr=pyo.quicksum(m.z.values()), sense=pyo.minimize)
m.cobertura = pyo.Constraint(expr=pyo.quicksum(m.x.values()) >= N_ASIGNADOS)


def paso2(eps, cobertura=True, extra=None):
    m.del_component("piso")
    m.piso = pyo.Constraint(expr=m.valor >= (1 - eps - 1e-7) * V_ESTRELLA)  # 1e-7: holgura numérica
    m.cobertura.activate() if cobertura else m.cobertura.deactivate()
    nombre = f"lex paso 2, ε = {eps:.2%}" + ("" if cobertura else ", SIN piso de cobertura") + (f", {extra}" if extra else "")
    return metricas(m, nombre, resolver(m))


for eps in [0.0, 0.001, 0.01]:
    resultados.append(paso2(eps))
resultados.append(paso2(0.01, cobertura=False))
tabla = pd.DataFrame(resultados)
tabla

**Lectura** (los detalles pueden cambiar con la versión de HiGHS si hay óptimos alternativos):
- El paso 1 da el valor máximo, pero la carga Junior va de **15 a 38**: nadie cuidó el balance.
- Con $\varepsilon = 0$: el mismo valor y $\Sigma z$ = 32, igual que el ponderado. Sin ceder nada, el balance ya mejora.
- Con $\varepsilon = 0{,}1\,\%$: $\Sigma z$ baja a **27** y los Junior quedan en **26–27**, casi iguales. Cuesta 0,1 % del valor.
- Con $\varepsilon = 1\,\%$: el balance **no mejora más** (27), pero la continuidad cae de 69,9 % a **61,7 %**. Ese punto está dominado por el de 0,1 %.
  El paso 2 solo mira $\Sigma z$; no cuida lo que no le pides.
- **La trampa:** sin el piso de cobertura, el modelo baja $\Sigma z$ a 16 **soltando 116 clientes más** (234 sin asignar). Le alcanza el 1 % de valor cedido para dejar clientes sin asesor.
  **Cada meta que importa necesita su restricción.**

### 6.3 Balance min-max: qué controla
$z_\sigma \ge$ carga de cada Junior: minimizar $z$ solo baja la carga **máxima**. Las cargas (50, 50, 10) y (50, 30, 30) tienen el mismo máximo: para el modelo valen igual.
Mira las cargas por Junior y segmento en el paso 1 y con $\varepsilon = 0{,}1\,\%$.

In [ ]:
def cargas_junior(eps):
    paso2(eps)
    x = np.array([pyo.value(m.x[a, c]) > 0.5 for a, c in zip(pares["asesor"], pares["cliente"])])
    asig = pares[x & (pares["tipo"] == "Junior").to_numpy()]
    return asig.pivot_table(index="asesor", columns="segmento", values="cliente", aggfunc="count", fill_value=0)


cargas = cargas_junior(0.0).join(cargas_junior(0.001), lsuffix=" ε=0", rsuffix=" ε=0,1%")
cargas

**Lectura:**
- Con $\varepsilon = 0$, el máximo por segmento queda en $z$, pero hay Junior muy por debajo: el min-max no los mira.
- Con $\varepsilon = 0{,}1\,\%$, las cargas quedan casi iguales en cada segmento.
- Si lo que importa es la **dispersión**, se penaliza la desviación respecto a la media: $\lvert \text{carga}_a - \bar{c} \rvert = d_a^+ + d_a^-$, con dos variables $\ge 0$.
  Y se puede balancear **saldo**, no solo número de clientes: 50 clientes de 1 millón no pesan lo mismo que 50 de 50 millones.

### 6.4 La frontera: valor contra balance
Con el ε-constraint en el paso 2 (cobertura protegida), varios $\varepsilon$. Esta celda tarda alrededor de 1 minuto.

In [ ]:
curva = pd.DataFrame([paso2(eps) for eps in [0.0, 0.0001, 0.0003, 0.0005, 0.001, 0.002]])
curva["valor_cedido_%"] = (100 * (1 - curva["valor_miles_de_millones"] / curva["valor_miles_de_millones"].iloc[0])).round(3)
curva[["version", "valor_cedido_%", "Σz", "carga_junior", "continuidad_%", "segundos"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva["valor_cedido_%"], curva["Σz"], "o-", color="#2563eb")
for x_, y_, etiqueta in zip(curva["valor_cedido_%"], curva["Σz"], curva["carga_junior"]):
    ax.annotate(etiqueta, (x_, y_), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set(xlabel="valor cedido (%)", ylabel="Σz (carga máx. Junior, 2 segmentos)",
       title="Cuánto valor cuesta el balance (etiqueta: carga Junior mín-máx)")
plt.tight_layout()
plt.show()

**Lectura para el jefe de cartera:**
- Sin ceder nada de valor, la suma de cargas máximas de los Junior baja de 42 (solo valor) a 32.
- Cediendo **0,05 % del valor**, baja a 27 y todos los Junior quedan con 26 a 27 clientes: carga pareja. Ceder más **no mejora** el balance:
  los puntos de 0,1 % y 0,2 % están dominados (mismo balance, menos valor, menos continuidad).
- La curva dice cuánto cuesta cada punto de balance: de 32 a 29, 0,011 % del valor; de 28 a 27, otro 0,02 %. La decisión es del área comercial, no del modelo.
- Hay que vigilar lo que el modelo no optimiza: la continuidad baja de 72,9 % a 71,8 % en el punto de 0,05 %. Si importa, va como restricción (ejercicio 3).
- Es la misma estructura que asignar técnicos a parques eólicos o ingenieros a proyectos: *same structure, different domain*.

## 7. ¿Qué método para varias metas?
Una sola regla para empezar:

> **Si negocio puede ordenar las metas, usa lexicográfico con tolerancia y protege cada meta que importa. Si no puede, muéstrale la frontera con ε-constraint y que elija.**

| Situación | Método | Por qué |
|---|---|---|
| Las metas tienen un precio de mercado común (todo es euros) | ponderado | los pesos son precios reales, no inventados |
| Hay un orden claro (valor primero, luego balance) | lexicográfico con $\varepsilon$ | sin pesos; cada meta en sus unidades |
| Negocio no sabe cuánto ceder | ε-constraint, mostrar la frontera | la curva es el menú |
| Una meta secundaria solo debe "desempatar" | ponderado con peso chico, con cuidado | el gap del solver puede tragarse el desempate |
| 3 o más metas | pisos en todas menos una | la frontera es una superficie: pocos puntos bien elegidos |
| Metas enteras, puntos del medio | nunca solo el ponderado | no llega a los puntos no soportados |

## 8. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Optimización multiobjetivo | Varias metas que chocan | VPN y CO₂; valor y balance | multi-objective optimization |
| Dominancia | Igual o mejor en todo, mejor en algo | 98 de 103 carteras dominadas | Pareto dominance |
| Frontera de Pareto | Las soluciones no dominadas | 5 carteras | Pareto front |
| Objetivo ponderado | Sumar metas con pesos | $\lambda$ de 0 a 1 | weighted sum |
| Normalizar | Dividir cada meta por su mejor valor | $f_k / f_k^*$ | normalization |
| Tasa de cambio escondida | Lo que vale una meta en unidades de otra | 54.000 € por kt | implicit trade-off |
| Punto no soportado | En la frontera, pero ningún peso lo encuentra | las 3 carteras del medio | unsupported Pareto point |
| Lexicográfico | Optimizar en orden, con tolerancia | valor y después balance | lexicographic optimization |
| Tolerancia | Cuánto se cede de la meta anterior | $\varepsilon = 0{,}1\,\%$ | tolerance, relaxation |
| Proteger metas | Piso para lo que importa y no se optimiza | piso de cobertura | goal protection |
| ε-constraint | Una meta en el objetivo, las demás como pisos | recorrer la frontera | epsilon-constraint method |
| Min-max | Minimizar la carga más alta | $\Sigma z$ | min-max balancing |

### Notación de hoy
| Símbolo | Significa | En la lección |
|---|---|---|
| $f_1(x)$, $f_2(x)$ | valor de cada meta | VPN, CO₂ |
| $x \succ x'$ | $x$ domina a $x'$ | 98 carteras dominadas |
| $\iff$, $\nexists$ | si y solo si; no existe | definición de la frontera |
| $\mathcal{P}$ | frontera de Pareto | 5 carteras |
| $f_k^*$ | mejor valor de la meta $k$ sola | 31,6 M€ y 582 kt |
| $\lambda$, $1 - \lambda$ | pesos del ponderado | 101 valores |
| $\varepsilon$, $(1 - \varepsilon) f_1^*$ | tolerancia y piso | 0,1 % del valor |
| $\text{lexmax}$ | máximo en orden de prioridad | 2 pasos |
| $\varepsilon_{k+1} = f_2(x_k) + \delta$ | el siguiente piso del ε-constraint | $\delta = 1$ kt |
| $\omega_a$, $\kappa_{ac}$ | peso del tipo, continuidad | cartera |
| $z_\sigma$ | carga máxima (min-max) | balance Junior |
| $\prod_{c} (k_c + 1)$ | espacio de búsqueda de una asignación | $10^{\dots}$ |

## 9. Ejercicios
1. **Un λ "razonable".** En la cartera de Banco Agrario sube $\lambda$ a 1.000.000 (la escala de una asignación). Antes, adivina: ¿cuánto valor cede y cuánto balance gana? ¿Qué punto de la curva de 6.4 da?
2. **Más presupuesto.** En la cartera de 8 proyectos, sube el presupuesto a 22 M€ y recorre la frontera con el ε-constraint. ¿Cuántos puntos tiene ahora? ¿El de máximo VPN cambia?
3. **Proteger la continuidad.** En el paso 2 con $\varepsilon = 1\,\%$, agrega un piso: continuidad $\ge$ la del paso 1 (`CONT_PASO1`). ¿Cuánto balance se pierde?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"How do you handle multiple objectives in an optimization model?"*
> *"I avoid hiding the trade-off in weights. A weighted sum fixes an implicit exchange rate, it can jump between extremes with small weight changes,
> and with integer variables it can't reach unsupported Pareto points: in my portfolio example, 101 different weights found only 2 of the 5 efficient portfolios.
> If the business can rank the goals, I solve lexicographically: maximize value, then keep value within a tolerance and minimize imbalance.
> If they can't, I run epsilon-constraint and show them the Pareto front."*

Seguimiento: *"Any pitfalls with the lexicographic approach?"*
> *"Two. First, the second step only optimizes what you ask: in my portfolio assignment, with 1% tolerance and no extra constraint, it dropped 116 clients
> to improve balance. Every goal that matters needs its own constraint, like a coverage floor. Second, scaling: the value floor should be written in sensible units,
> like millions, so the constraint is numerically stable."*

Seguimiento: *"Your balance constraint is min-max. Is that enough?"*
> *"Min-max only controls the peak; loads of 50, 50, 10 and 50, 30, 30 look the same. If the spread matters, I penalize deviations from the average load,
> and I'd balance value, not only client counts."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
m1 = construir()
m1.obj = pyo.Objective(expr=m1.valor - 1_000_000 * pyo.quicksum(m1.z.values()), sense=pyo.maximize)
pd.DataFrame([metricas(m1, "ponderado λ = 1.000.000", resolver(m1)), tabla.iloc[0].to_dict(), tabla.iloc[3].to_dict()])

Con $\lambda = 1.000.000$, $\Sigma z$ baja a **28** (cargas 22 a 28) y cede 0,016 % del valor. No suelta clientes: el valor de una asignación (unos 4,9 millones) sigue pesando más que una unidad de balance.
Es un punto de la curva de 6.4, entre el de 0,011 % (29) y el de 0,026 % (28). Pero nadie eligió ese punto: salió de un $\lambda$ escogido a ojo.
Con $\lambda = 1.000$ el balance solo desempata; con $10^6$ ya pesa; con $10^7$ podría empezar a soltar clientes. **El resultado depende de un número que nadie sabe justificar.**
El lexicográfico hace la misma pregunta en unidades que el banco entiende: "¿cuánto valor cedes?".

In [ ]:
# Ejercicio 2
PRESUPUESTO = 22
puntos22, piso_co2 = [], 0
while True:
    r = cartera("vpn", pisos={"co2": piso_co2})
    if r is None:
        break
    r = cartera("co2", pisos={"vpn": r["vpn"], "co2": piso_co2})
    puntos22.append({"cartera": r["cartera"], "vpn": r["vpn"], "co2": r["co2"]})
    piso_co2 = r["co2"] + 1
PRESUPUESTO = 19
pd.DataFrame(puntos22)

Con 22 M€ la frontera tiene **6 carteras** en vez de 5, y está más arriba y más a la derecha: el máximo VPN sube de 31,6 a **35,4 M€** (eólico A + solar D + repotenciación F + subestación G).
La repotenciación F y el solar D están en **todas** las carteras de la frontera, con los dos presupuestos: son las decisiones más robustas. Eso también se le dice al directorio.
Es sensibilidad por re-solve (lección 3): cambias el dato y vuelves a recorrer la frontera.

In [ ]:
# Ejercicio 3
m.del_component("piso_continuidad")
m.piso_continuidad = pyo.Constraint(expr=m.continuidad >= CONT_PASO1)
con_continuidad = paso2(0.01, extra="continuidad protegida")
m.del_component("piso_continuidad")
pd.DataFrame([tabla.iloc[4].to_dict(), con_continuidad])

Con la continuidad protegida, el balance **no empeora**: $\Sigma z$ sigue en 27, y los Junior quedan todos con 27 clientes. La continuidad vuelve a **72,9 %** (antes 61,7 %) y el valor baja apenas, de 18,682 a 18,680 miles de millones.
El punto de $\varepsilon = 1\,\%$ sin protección había bajado la continuidad **sin necesidad**: el modelo no la miraba, así que cualquier solución con $\Sigma z = 27$ le daba igual.
Tarda más (unos 20 s): el piso extra hace más difícil el MILP. **Regla:** cada meta que importa, o está en el objetivo, o tiene un piso.